In [42]:
import sys, os
import glob
import pandas as pd
import torch
import torch.nn as nn
sys.path.append("/home/jgershon/git/cleo")
import pdb_util

def seq_to_onehot(seq):
    input_feat = torch.tensor([pdb_util.aa12num[x] for x in seq])
    input_feat = torch.nn.functional.one_hot(input_feat, num_classes=20)
    return input_feat.float()

data = pd.read_csv("/home/jgershon/projects/itopt/kiera/g20_optimization/251209_processed_filtered_data_round0_round1_with_zscore_with_val.csv")

val_data = data[data["val"]]
train_data = data[~data["val"]]


# get training data
train_x = torch.stack([seq_to_onehot(seq) for seq in train_data["sequence"]])
train_x = train_x.reshape(train_x.shape[0], -1) # flatten final two dims

train_y = torch.tensor(train_data["z_score_norm_rate"].values).unsqueeze(-1)

In [34]:
class PottsModel(nn.Module):
    
    # D is the flattened sequence dimention, IE L*20
    def __init__(self, D, use_two_body=True):
        super().__init__()
        self.D = D
        self.use_two_body = use_two_body

        # One-body fields
        self.h = nn.Parameter(torch.zeros(self.D))
        nn.init.normal_(self.h, mean=0.0, std=0.01) # init with small values close to zero

        if self.use_two_body:
            # Two-body couplings
            self.J = nn.Parameter(torch.zeros(self.D, self.D))
            nn.init.normal_(self.J, mean=0.0, std=0.01) # init with small values close to zero



    def forward(self, x):
        """
        x: [B, D] already flattened one-hot encoded sequences
        returns: [B, 1] predicted activity (energy)
        """
        B = x.shape[0]

        # One-body term: [B]
        energy = torch.sum(x * self.h, dim=1)


        if not self.use_two_body:
            # Two-body term:
            #
            # compute x^T J x for each batch
            # (B, D) @ (D, D) -> (B, D) -> elementwise mul and sum over D
            xJ = x @ self.J            # [B, D]
            energy += torch.sum(xJ * x, dim=1)

        # Output [B, 1]
        return energy.unsqueeze(1)


In [35]:
model = PottsModel(D=train_x.shape[1], use_two_body=True)

In [37]:
x = train_x[:16]

out = model(x)

In [41]:
train_x.shape

torch.Size([1913, 4180])